In [27]:
import pandas as pd
import json
import numpy as np
import sys
sys.path.append("../")
from utils import load_item_attributes
pd.set_option('display.max_colwidth', 100)
from tqdm import tqdm, tqdm_notebook
tqdm_notebook().pandas()

/tmp/ipykernel_131087/2875277136.py:9: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  tqdm_notebook().pandas()


0it [00:00, ?it/s]

In [ ]:
config = {
    "test_data": "data/citeulike/test_data.parquet",
    "attribute_path": "dataset/Citeulike/item_desciption_small.json",
    "pred_df": "outputs/citeulike/VLGCN_prediction_desciption_wTargetProfile.parquet",
    "api_key": "",
    "togetherAI_api_key": ""
}


In [29]:
df = pd.read_parquet(config["test_data"])
item_attributes = load_item_attributes(config["attribute_path"])

In [30]:
test_df = pd.read_parquet(config['pred_df'])

# LLM as jugde

In [34]:
eval_prompt_template = """
# INSTRUCTION
Please act as an impartial judge and evaluate the AI assistant’s recommendation decision as well as decision explanation based on the user’s purchase history, target item, and ground truth label. Assign a score according to the following four levels:  
RATING-0: Incorrect classification - The assistant fails to generate a correct recommendation decision.
RATING-1: Correct classification, insufficient explanation - The assistant correctly makes the recommendation decision but provides no, few, or irrelevant explanations, or provides explanations with hallucination, some of which do not conform to the actual situation.
RATING-2: Correct classification, acceptable explanation - The assistant correctly makes the recommendation decision and provides an explanation that is logically consistent and aligns with the user’s history and target item. But the explanation still has minor imperfections such as lack of persuasiveness or informativeness.
RATING-3: Correct classification, satisfying explanation - The assistant correctly makes the recommendation decision and provides a satisfactory explanation, including a summary of the user’s historical behavior patterns and characteristics, as well as a thorough analysis of the consistency or inconsistency between user preferences and the target item.  
Please give your score in the form of <br>RATING</br>, for example, if the rating is 1, output <br>RATING-1</br>. Do not allow the length of the explanation to influence your evaluation. Be as objective as possible.  

# INPUT
nown information: 
User history: {USER HISTORY}, 
Target item: {ITEM}, 
Label: {LABEL}. 
Assistant’s output: 
{EXPLANATIONS}
"""

In [35]:
def construct_eval_prompt(row):
    user_hist_item = row['neighbor_item']
    target_item_id = row['item_id']
    label = "Yes" if row['label'] == True else "No" if row['label'] == False else label
    explaination = row['reasoning']
    
    user_history = []
    for item_id in user_hist_item:
        user_history.append("Name: " + json.loads(item_attributes[item_id])['Title'] + "\n" + "description: " +  json.loads(item_attributes[item_id])['description'])
    
    target_item = "Name: " + json.loads(item_attributes[target_item_id])['Title'] + "\n" + "description: " +  json.loads(item_attributes[target_item_id])['description']
    
    eval_prompt = eval_prompt_template.replace("{USER HISTORY}", "\n".join(user_history))\
                                        .replace("{ITEM}", target_item)\
                                        .replace("{LABEL}", str(label))\
                                        .replace("{EXPLANATIONS}", explaination)
    return eval_prompt

In [36]:
def parse_rating(x):
    for i in ['RATING-0','RATING-1','RATING-2','RATING-3']:
        if i in x:
            return i
    return 'n/a'

## Gemini

In [37]:
from google import genai
import time
client = genai.Client(api_key=config["api_key"])
from google.genai import types

In [38]:
def gemini_batch_inferences(df, temp_file_name, display_name, model_name, max_token = 8):
    requests = []
    for idx, row in df.iterrows():
        resq = {
            "key": f"{row['user_id']}_{row['item_id']}",
            "request": {
                "contents": [{
                    "parts": [{
                        "text": row['eval_prompt']
                    }]
                }]
            }
        }
        requests.append(resq)
    
    with open(temp_file_name, "w") as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")
    
    
    uploaded_file = client.files.upload(
        file=temp_file_name,
        config=types.UploadFileConfig(display_name=display_name, mime_type='jsonl')
    )
    
    print(f"Uploaded file: {uploaded_file.name}")

    file_batch_job = client.batches.create(
        model="gemini-2.5-flash-lite",
        src=uploaded_file.name,
        config={
            'display_name': display_name,
        },
    )
    print(f"Created batch job: {file_batch_job.name}")

    # Use the name of the job you want to check
    # e.g., inline_batch_job.name from the previous step
    job_name = file_batch_job.name # (e.g. 'batches/your-batch-id')
    batch_job = client.batches.get(name=job_name)

    
    
    completed_states = set([
        'JOB_STATE_SUCCEEDED',
        'JOB_STATE_FAILED',
        'JOB_STATE_CANCELLED',
        'JOB_STATE_EXPIRED',
    ])
    
    print(f"Polling status for job: {job_name}")
    batch_job = client.batches.get(name=job_name) # Initial get
    while batch_job.state.name not in completed_states:
      print(f"Current state: {batch_job.state.name}")
      time.sleep(30) # Wait for 30 seconds before polling again
      batch_job = client.batches.get(name=job_name)
    
    print(f"Job finished with state: {batch_job.state.name}")
    if batch_job.state.name == 'JOB_STATE_FAILED':
        print(f"Error: {batch_job.error}")
    
    if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    
        # If batch job was created with a file
        if batch_job.dest and batch_job.dest.file_name:
            # Results are in a file
            result_file_name = batch_job.dest.file_name
            print(f"Results are in file: {result_file_name}")
    
            print("Downloading result file content...")
            file_content = client.files.download(file=result_file_name)
            # Process file_content (bytes) as needed
            print(file_content.decode('utf-8'))
            # Parse the JSONL string into a list of dictionaries
            parsed_responses = [
                json.loads(line) for line in file_content.decode('utf-8').strip().split('\n')
            ]
    
        # If batch job was created with inline request
        elif batch_job.dest and batch_job.dest.inlined_responses:
            # Results are inline
            print("Results are inline:")
            for i, inline_response in enumerate(batch_job.dest.inlined_responses):
                print(f"Response {i+1}:")
                if inline_response.response:
                    # Accessing response, structure may vary.
                    try:
                        print(inline_response.response.text)
                    except AttributeError:
                        print(inline_response.response) # Fallback
                elif inline_response.error:
                    print(f"Error: {inline_response.error}")
        else:
            print("No results found (neither file nor inline).")
    else:
        print(f"Job did not succeed. Final state: {batch_job.state.name}")
        if batch_job.error:
            print(f"Error: {batch_job.error}")
    df['response'] = parsed_responses
    return df

In [ ]:
test_df = pd.read_parquet(config['pred_df']).drop(columns=['label'])
test_df = df.merge(test_df, on = ['user_id','item_id'])
test_df['eval_prompt'] = test_df.apply(lambda x: construct_eval_prompt(x), axis=1)

In [ ]:
exp_name=config['pred_df'].split('/')[-2] + '_' + config['pred_df'].split('/')[-1].split('.')[-2]
res_df = gemini_batch_inferences(test_df, f"temp/EvalGemini25Flash_{exp_name}.jsonl", exp_name, "gemini-2.5-flash-lite")

In [ ]:
res_df['eval_score'] = res_df.response.map(lambda x: parse_rating(x['response']['candidates'][0]['content']['parts'][0]['text']))
res_df["score"] = res_df.eval_score.map(
    lambda x: int(x.strip("RATING-")) if isinstance(x, str) and x.strip("RATING-").isdigit() else 0
)
res_df['new_pred'] = res_df['pred'].apply(lambda x: x == 'LIKE')
res_df['score'] = res_df.apply(lambda x: 1 if ((x['label'] == x['new_pred']) and (x['score'] == 0)) else x['score'], axis=1)

In [43]:
res_df.to_parquet(f"LLMEval/EvalGemini25Flash_{exp_name}.parquet")

## Together AI

In [48]:
from together import Together
import time
client = Together(api_key=config['togetherAI_api_key'])

In [49]:
def together_ai_inferences(df, temp_file_name, model_name, max_token = 8):
    df['custom_id'] = df.index.map(lambda x: str(hash(x)))
    temp_file_name = temp_file_name
    requests = []
    for idx, row in df.iterrows():
        resq = {
            "custom_id": row['custom_id'],
            "body": {
                "model": model_name,
                "messages": [
                    {
                        "role": "user",
                        "content": row['eval_prompt']
                    }
                ],
                "max_tokens":max_token,
                "chat_template_kwargs": {"enable_thinking": "false"},
                "reasoning_effort": "low"
            }
        }
        requests.append(resq)

    with open(temp_file_name, "w") as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")
    
    file_resp = client.files.upload(file=temp_file_name, purpose="batch-api")
    file_id = file_resp.id
    print(f"file id {file_id}")
    batch = client.batches.create_batch(file_id, endpoint="/v1/chat/completions")
    completed_states = ["COMPLETED", "FAILED", "EXPIRED", "CANCELLED"]
    print(f"Polling status for job: {batch.id}")
    batch_stat = client.batches.get_batch(batch.id)
    while batch_stat.status not in completed_states:
        print(f"Current state: {batch_stat.status}")
        time.sleep(30) # Wait for 30 seconds before polling again
        batch_stat = client.batches.get_batch(batch.id)
    
    print(f"Job finished with state: {batch_stat.status}")
    if batch_stat.status == 'FAILED':
        print(f"Error: {batch_stat.error}")
    if batch_stat.status == 'COMPLETED':
        print("Downloading result file content...")
        file_content = client.files.retrieve_content(id=batch_stat.output_file_id, output="output_" + temp_file_name.replace(".jsonl", "_output.jsonl"))
    
    res_df = {"custom_id": [], "response": []}
    with open(file_content.filename, 'r') as f:
        for line in f.readlines():
            line = json.loads(line)
            res_df["custom_id"].append(line['custom_id'])
            res_df["response"].append(line['response']['body']['choices'][0]['message']['content'])
    res_df = pd.DataFrame.from_dict(res_df)
    res_df = df.merge(res_df, on = 'custom_id')
    return res_df



In [ ]:
test_df = pd.read_parquet(config['pred_df']).drop(columns=['label'])
test_df = df.merge(test_df, on = ['user_id','item_id'])
test_df['eval_prompt'] = test_df.apply(lambda x: construct_eval_prompt(x), axis=1)

In [51]:
test_df['eval_prompt'] = test_df.apply(lambda x: construct_eval_prompt(x), axis=1)

In [ ]:
exp_name=config['pred_df'].split('/')[-2] + '_' + config['pred_df'].split('/')[-1].split('.')[-2]
res_df = together_ai_inferences(test_df, f"temp/EvalDeepSeek_{exp_name}.jsonl", "deepseek-ai/DeepSeek-V3")

In [ ]:
res_df['eval_score'] = res_df.response.map(parse_rating)

In [ ]:
res_df["score"] = res_df.eval_score.map(
    lambda x: int(x.strip("RATING-")) if isinstance(x, str) and x.strip("RATING-").isdigit() else 0
)
res_df['new_pred'] = res_df['pred'].apply(lambda x: x == 'LIKE')
res_df['score'] = res_df.apply(lambda x: 1 if ((x['label'] == x['new_pred']) and (x['score'] == 0)) else x['score'], axis=1)

In [ ]:
res_df.to_parquet(f"LLMEval/EvalDeepSeek_{exp_name}.parquet")